In [ ]:
from pathlib import Path

import pandas as pd
from rdkit import Chem

In [ ]:
file_path = r"../pahdb_w24146_neutral.csv"

In [ ]:
def count_carbon_atoms(smi):
    """
    Counts carbon atoms in a SMILES string with error handling for invalid structures.

    Args:
        smi: SMILES string to be parsed.

    Returns:
        int: Number of carbon atoms (atomic number 6), or None if parsing fails.
    """

    try:
        # Convert SMILES to RDKit molecule object
        mol = Chem.MolFromSmiles(smi)
        if mol:
            # Atomic number 6 represents Carbon (C)
            carbon_count = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)
            return carbon_count
        return None
    except Exception:
        # Gracefully handle RDKit C++ level errors or unexpected input types
        return None

In [ ]:
# 1. Load the raw dataset containing chemical SMILES
df = pd.read_csv(Path(file_path))
print(f"Original dataset shape: {df.shape}")

# 2. Extract carbon count feature using RDKit-based logic
df['n_c'] = df['canonical_smiles'].apply(count_carbon_atoms)

# 3. Clean the dataset by removing failed RDKit parses (None values)
# Ensures that only chemically valid sequences enter the training pipeline
df = df.dropna(subset=['n_c'])

# 4. Persist the refined dataset for downstream tasks
df.to_csv(file_path, index=False)
print(f"Done.")

# 5. Segment the data based on molecular size (Carbon count)
# Small to medium PAHs vs. Large PAH clusters
less_than_100 = df[df['n_c'] < 100].shape[0]
greater_equal_100 = df[df['n_c'] >= 100].shape[0]

# 6. Summary statistics for data audit
print('-' * 30)
print(f"Final dataset shape: {df.shape}")
print(f"Molecules with Carbon count < 100: {less_than_100}")
print(f"Molecules with Carbon count >= 100: {greater_equal_100}")